# Day 2 — Dynamics exploration

Four experiments against the verified 3-DoF model. The point is not to write code —
it is to extract physical numbers you can defend in an interview.

| | |
|---|---|
| **A** | Find the suicide-burn altitude |
| **B** | Show the vehicle physically cannot hover |
| **C** | Propellant budget and margin |
| **D** | Break the integrators with a huge step |

In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('..'))

from src.dynamics import (Vehicle, dynamics_3dof, control_zero,
                          control_constant, G0, G_EARTH)
from src.integrators import propagate

veh = Vehicle()
DT = 0.002          # fine enough that the vz=0 crossing is well resolved
print(veh.summary())

## Experiment A — the suicide burn altitude

Enter at −200 m/s, apply full thrust, and find where the vehicle comes to rest.
Then solve for the entry altitude that brings it to rest at exactly `z = 0`.

In [ ]:
def burn_to_stop(h_start, vz_start, m_start=None):
    """Full vertical thrust from the given state. Returns (altitude, mass) at vz = 0."""
    m_start = veh.m_wet if m_start is None else m_start
    y0 = np.array([0.0, h_start, 0.0, vz_start, m_start])
    t, y = propagate(dynamics_3dof, y0, (0.0, 60.0), DT,
                     control_constant(0.0, veh.T_max), veh, method='rk4')
    vz = y[:, 3]
    i = np.argmax(vz >= 0.0)
    if vz[i] < 0:
        return None, None                      # never arrested the descent
    if i == 0:
        return y[0, 1], y[0, 4]
    f = -vz[i-1] / (vz[i] - vz[i-1])           # interpolate onto the crossing
    return (y[i-1, 1] + f*(y[i, 1] - y[i-1, 1]),
            y[i-1, 4] + f*(y[i, 4] - y[i-1, 4]))


for h in (5000, 3000, 2000, 1000):
    z, m = burn_to_stop(h, -200.0)
    print(f'start {h:>5} m -> stops at {z:8.1f} m   '
          f'(stopping distance {h-z:6.1f} m, prop {veh.m_wet-m:5.0f} kg)')

Notice the **stopping distance is identical** in every case. Of course it is — the
burn starts from the same speed and mass every time, so it is the same manoeuvre
translated up and down. The suicide burn is therefore defined by a *distance*, not
an altitude, and the altitude falls out of whatever speed you happen to arrive with.

Bisect for the entry altitude that lands it exactly on the pad:

In [ ]:
lo, hi = 100.0, 5000.0
for _ in range(60):
    mid = 0.5*(lo + hi)
    z, _ = burn_to_stop(mid, -200.0)
    hi, lo = (mid, lo) if z > 0 else (hi, mid)

h_ideal = 0.5*(lo + hi)
z_chk, m_chk = burn_to_stop(h_ideal, -200.0)
print(f'ideal burn altitude at -200 m/s : {h_ideal:.1f} m')
print(f'  stops at                      : {z_chk:.4f} m')
print(f'  propellant used               : {veh.m_wet-m_chk:,.0f} kg')

### The realistic version

In reality the vehicle is *accelerating* while it waits. Free-fall from 5 km at
−200 m/s, ignite at altitude `h_b`, and solve for the `h_b` that lands it.

In [ ]:
y0 = np.array([0.0, 5000.0, 0.0, -200.0, veh.m_wet])
t_ff, y_ff = propagate(dynamics_3dof, y0, (0.0, 60.0), DT,
                       control_zero, veh, method='rk4')

def freefall_then_burn(h_b):
    i = np.argmax(y_ff[:, 1] <= h_b)
    if y_ff[i, 1] > h_b:
        return None, None
    return burn_to_stop(y_ff[i, 1], y_ff[i, 3], y_ff[i, 4])

lo, hi = 200.0, 4900.0
for _ in range(60):
    mid = 0.5*(lo + hi)
    z, _ = freefall_then_burn(mid)
    hi, lo = (mid, lo) if (z is None or z > 0) else (hi, mid)

h_b = 0.5*(lo + hi)
z_b, m_b = freefall_then_burn(h_b)
i = np.argmax(y_ff[:, 1] <= h_b)
print(f'ignition altitude   : {h_b:.1f} m')
print(f'speed at ignition   : {abs(y_ff[i, 3]):.1f} m/s')
print(f'touchdown altitude  : {z_b:.3f} m')
print(f'propellant used     : {veh.m_wet-m_b:,.0f} kg  '
      f'({100*(veh.m_wet-m_b)/veh.m_prop_initial:.0f}% of the landing load)')

## Experiment B — the vehicle cannot hover

Thrust-to-weight at **minimum** throttle is the number that matters. If it exceeds
1, the engines cannot be throttled down far enough to balance weight.

In [ ]:
print(f'{"case":<12}{"mass [kg]":>12}{"TWR_min":>10}{"net accel":>12}')
for label, m in (('wet mass', veh.m_wet),
                 ('dry + 5 t', veh.m_dry + 5000),
                 ('dry mass', veh.m_dry)):
    print(f'{label:<12}{m:>12,.0f}{veh.T_min/(m*G_EARTH):>10.2f}'
          f'{veh.T_min/m - G_EARTH:>+11.2f} m/s^2')

y0 = np.array([0.0, 500.0, 0.0, 0.0, veh.m_dry + 5000.0])
t, y = propagate(dynamics_3dof, y0, (0.0, 5.0), DT,
                 control_constant(0.0, veh.T_min), veh, method='rk4')
print(f'\nHover attempt at minimum throttle, 5 t of propellant left:')
print(f'  after 5 s: {y[-1,1]:.1f} m (started at 500.0), climbing {y[-1,3]:+.1f} m/s')

plt.figure(figsize=(7,4))
plt.plot(t, y[:,1], lw=2)
plt.axhline(500, ls='--', c='k', alpha=0.5, label='intended hover altitude')
plt.xlabel('Time [s]'); plt.ylabel('Altitude [m]')
plt.title('Minimum throttle is still an ascent')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

**The insight.** TWR at minimum throttle is 2.16 even at *wet* mass, rising to 2.81
at dry mass. It is above 1 for the entire landing burn, so this vehicle can never
hover — not merely "not near touchdown".

That is why the landing has to be a single, precisely-timed burn that arrives at
zero velocity exactly at zero altitude. There is no hover-and-descend option and no
second attempt. It is a consequence of engine sizing, not a design preference.

## Experiment C — propellant budget

Under ideal hover, thrust equals weight, so `dm/dt = -m/Isp`. Mass decays
exponentially and the endurance has a closed form: `t = Isp · ln(m_wet/m_dry)`.

In [ ]:
def hover_ctrl(t, state, vehicle):
    return np.array([0.0, state[4]*G_EARTH])

t_analytic = veh.isp*np.log(veh.m_wet/veh.m_dry)

t, y = propagate(dynamics_3dof, np.array([0.0, 1000.0, 0.0, 0.0, veh.m_wet]),
                 (0.0, 120.0), 0.01, hover_ctrl, veh, method='rk4')
i = np.argmax(y[:,4] <= veh.m_dry)

print(f'analytic endurance  : {t_analytic:.1f} s')
print(f'simulated endurance : {t[i]:.1f} s')
print(f'a real landing burn is ~15-20 s -> margin ~{t_analytic/17.5:.1f}x')

## Experiment D — break it

Free-fall is a quadratic in time, and RK4 integrates polynomials up to degree 4
*exactly*. So on this problem RK4 is at machine precision no matter how absurd the
step size, while Euler degrades linearly.

In [ ]:
z0, vz0, t_end = 5000.0, -100.0, 20.0
print(f'{"dt [s]":>8}{"Euler error [m]":>20}{"RK4 error [m]":>18}')
for dt in (2.0, 1.0, 0.5, 0.25):
    errs = []
    for method in ('euler', 'rk4'):
        t, y = propagate(dynamics_3dof, np.array([0.0, z0, 0.0, vz0, veh.m_wet]),
                         (0.0, t_end), dt, control_zero, veh, method=method)
        exact = z0 + vz0*t[-1] - 0.5*G_EARTH*t[-1]**2
        errs.append(abs(y[-1,1] - exact))
    print(f'{dt:>8}{errs[0]:>20.4f}{errs[1]:>18.3e}')

At `dt = 2 s` Euler is wrong by **196 m** — it would have you land inside the pad.
RK4 is wrong by 2e-12 m.

Do not over-generalise this one: RK4 is *exact* here only because free-fall is a
low-degree polynomial. Experiment 4 in the test suite uses variable mass, where the
solution is logarithmic, and there RK4 shows its true fourth-order behaviour.